# 🚀 Workshop Day 1: Data Engineering — Turning Junk into Fuel
## Agentic RAG: From Zero to Hero

**Duration:** 3 hours | **Level:** Undergraduate

---

### 🎯 Goals for Today

We will learn the end-to-end **Data Engineering Pipeline** for RAG:

```
📄 Raw documents → 🔍 Duplicate check → ✂️ Chunking → 📝 Markdown → 🔢 Embedding → 🗄️ VectorDB → 🔎 Retrieval
```

> **RAG (Retrieval-Augmented Generation)** is a technique that helps LLMs answer more accurately
> by first "retrieving" relevant information before "generating" the answer.



### 🗺️ Pipeline 

| Step | Section | Task |
|:----:|---------|------|
| 📁 | Input | Raw Documents (PDF, TXT, DOCX) |
| ⬇️ | 1.1 Deduplication | MD5 Hash |
| ⬇️ | 1.2 Chunking | |
| ⬇️ | 1.3-1.4 Markdown | PDF → Markdown (Gemini / Docling) |
| ⬇️ | 1.5 Embedding | → Vector (multilingual-e5-large) |
| ⬇️ | 1.6 Hybrid Search | Dense + Sparse (BM25) |
| ⬇️ | 1.7 VectorDB | Qdrant |
| ⬇️ | 1.8 Indexing | Upsert vectors Qdrant |
| 🔎 | 1.9 Retrieval | vectors query |

> 💡 **** — RAG !

---
## 📦 Section 0: Dependencies

 library workshop 

In [16]:
%%time
# libraries 
!pip install -q google-genai \
 'docling>=2.31' \
 'docling-ibm-models>=3.4' \
 sentence-transformers \
 qdrant-client \
 langchain-text-splitters \
 rank_bm25 \
 pymupdf \
 pythainlp \
 scikit-learn \
 rich

# cache docling-models ONNX file mismatch
import shutil, os
cache_path = os.path.expanduser('~/.cache/huggingface/hub/models--ds4sd--docling-models')
if os.path.exists(cache_path):
 shutil.rmtree(cache_path)
 print('🗑️ docling-models cache ')

print('✅ !')
print('⚠️ Restart runtime: Runtime → Restart session → Run cell ')

 Preparing metadata (setup.py) ... done
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.4/397.4 kB 25.6 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 6.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 26.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 71.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 63.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 18.6 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 85.3 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.6 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.9 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 81.1 MB/s eta 0:00:00
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
%%time
# Import libraries 
import hashlib
import os
import json
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

print('✅ Import !')

✅ Import !
CPU times: user 0 ns, sys: 995 µs, total: 995 µs
Wall time: 1.89 ms


### 📂 Prepare Sample Data

We will create sample Thai-language data to use throughout the workshop.



In [18]:
%%time
# 
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

# : Case Study AI 
sample_texts = {

 'case_kmitl.txt': ''': AI (KMITL)

 KMITL AI Smart Campus
 IoT sensor Machine Learning 
 25% 

 NLP 

 Sentiment Analysis Topic Modeling 


 KMITL RAG AI Tutor
 200 
 24 
 lecture notes, slides, 
''',

 'case_healthcare.txt': ''': AI 

 AI (Medical Imaging)
 X-ray Deep Learning
 AI 95% 92%

 NLP (EMR)

 15 2 

: 
 Transfer Learning 
 Fine-tune ''',

 'case_banking.txt': ''': AI 

 Chatbot KBTG
 Large Language Model (LLM) RAG


 RAG 
 FAQ
 LLM 

: Call Center 40%
 25%
 24 

: Vector Database Embedding
Hybrid Search Dense + Sparse ''',

 'case_banking_duplicate.txt': ''': AI 

 Chatbot KBTG
 Large Language Model (LLM) RAG


 RAG 
 FAQ
 LLM 

: Call Center 40%
 25%
 24 

: Vector Database Embedding
Hybrid Search Dense + Sparse ''',

 'case_education.txt': ''': AI 

 AI 
 Intelligent Tutoring System 

 RAG -
 Slides 
 Embed Vector Vector Database

 
 LLM 

:
1. : PDF Markdown
2. : Chunking 
3. Embedding: Chunk Vector 
4. : Upsert Vector Qdrant Vector DB 
5. : Hybrid Search Chunk 
6. : Context LLM 

: 24 
 60%''',

 'case_agriculture.txt': ''': AI 

Smart Farming AI 
 IoT Sensor 

 Computer Vision 
 Convolutional Neural Network (CNN) Train 
 50,000 8 

 NLP 
 Social Media


 AI :
- 
- 
- Data Engineering '''
}

for fname, content in sample_texts.items():
 with open(f'data/{fname}', 'w', encoding='utf-8') as f:
 f.write(content)

print(f'✅ {len(sample_texts)} data/')
print()
for fname in sorted(sample_texts.keys()):
 print(f' 📄 {fname}')

✅ 5 data/

 📄 case_agriculture.txt
 📄 case_banking.txt
 📄 case_banking_duplicate.txt
 📄 case_education.txt
 📄 case_healthcare.txt
CPU times: user 12 µs, sys: 999 µs, total: 1.01 ms
Wall time: 2.05 ms


---
## 🔍 Section 1.1: Duplicate MD5

### MD5 Hash ?

**MD5 (Message Digest Algorithm 5)** hash 128-bit (32 hex)

- → hash ****
- → hash ****

### Duplicate?

 RAG :
- ❌ storage embedding
- ❌ 
- ❌ LLM 

In [19]:
%%time
def compute_md5(filepath):
 """ MD5 hash """
 md5_hash = hashlib.md5()
 with open(filepath, 'rb') as f:
 for chunk in iter(lambda: f.read(4096), b''):
 md5_hash.update(chunk)
 return md5_hash.hexdigest()

# : MD5 
print('📊 MD5 Hash :')
print('=' * 70)
file_hashes = {}
for fname in sorted(os.listdir('data')):
 fpath = f'data/{fname}'
 if os.path.isfile(fpath):
 h = compute_md5(fpath)
 file_hashes[fname] = h
 print(f' {fname:<25} → {h}')

print('\n🔍 : case_banking.txt case_banking_duplicate.txt hash !')

📊 MD5 Hash :
 case_agriculture.txt → 1d03b92b2dbb2d89eeb0c053207fe283
 case_banking.txt → 66160d60bd9880b6b155915e9f0b7c1d
 case_banking_duplicate.txt → 66160d60bd9880b6b155915e9f0b7c1d
 case_education.txt → 8e47b92d1afdb2f61800890133ffe7f7
 case_healthcare.txt → 2c56ffd8024fe12efc351c587515339d

🔍 : case_banking.txt case_banking_duplicate.txt hash !
CPU times: user 0 ns, sys: 942 µs, total: 942 µs
Wall time: 1.76 ms


In [20]:
%%time
def find_duplicates(directory):
 """ MD5"""
 hash_map = {} # md5 → [list of filepaths]

 for fname in os.listdir(directory):
 fpath = os.path.join(directory, fname)
 if os.path.isfile(fpath):
 h = compute_md5(fpath)
 hash_map.setdefault(h, []).append(fname)

 # 1 
 duplicates = {h: files for h, files in hash_map.items() if len(files) > 1}
 return duplicates, hash_map

duplicates, hash_map = find_duplicates('data')

if duplicates:
 print('⚠️ !')
 for h, files in duplicates.items():
 print(f'\n Hash: {h}')
 print(f' : {", ".join(files)}')
 print(f' → : {files[0]}, : {", ".join(files[1:])}')
else:
 print('✅ ')

⚠️ !

 Hash: 66160d60bd9880b6b155915e9f0b7c1d
 : case_banking_duplicate.txt, case_banking.txt
 → : case_banking_duplicate.txt, : case_banking.txt
CPU times: user 1 ms, sys: 0 ns, total: 1 ms
Wall time: 7.06 ms


In [21]:
%%time
# ()
removed = []
for h, files in duplicates.items():
 for f in files[1:]: # 
 os.remove(f'data/{f}')
 removed.append(f)

print(f'🗑️ {len(removed)} : {", ".join(removed)}')
print(f'📁 : {sorted(os.listdir("data"))}')

🗑️ 1 : case_banking.txt
📁 : ['case_agriculture.txt', 'case_banking_duplicate.txt', 'case_education.txt', 'case_healthcare.txt']
CPU times: user 0 ns, sys: 900 µs, total: 900 µs
Wall time: 2.41 ms


### 💡 
- `case_banking.txt` `case_banking_duplicate.txt` **hash ** → 100%
- MD5 **** 
- duplicate → storage + 

> 🎯 — MD5 

### 🧪 Exercise 1.1

Try creating 2–3 additional files (with some duplicates), then use `find_duplicates()` to verify the results.



In [ ]:
%%time
# 🧪 : duplicate detection

# 💡 Hint:
# 1. open('data/my_file.txt', 'w')
# 2. 
# 3. find_duplicates('data') 

# with open('data/test1.txt', 'w') as f:
# f.write('')
# with open('data/test2.txt', 'w') as f:
# f.write('') # !
# find_duplicates('data')

---
## ✂️ Section 1.2: Chunking 

### Chunk?

- LLM **context window ** — 100 
- Embedding model **** ( 512 tokens)
- Chunk **retrieval **

### Chunk 

| | | | |
|------|---------|-------|--------|
| Fixed-size | | , | |
| Recursive | separator | fixed | separators |
| Sentence-based | | | chunk |
| Semantic | embedding | | , |

In [22]:
%%time
# 
all_docs = {} # 
all_text = ''

for fname in sorted(os.listdir('data')):
 fpath = f'data/{fname}'
 if os.path.isfile(fpath) and fname.endswith('.txt'):
 with open(fpath, 'r', encoding='utf-8') as f:
 content = f.read()
 all_docs[fname] = content
 all_text += '\n\n' + content

print(f'📄 : {len(all_docs)} ')
print(f'📄 : {len(all_text)} ')
for fname, content in all_docs.items():
 print(f' 📄 {fname}: {len(content)} ')

📄 : 4 
📄 : 2628 
 📄 case_agriculture.txt: 629 
 📄 case_banking_duplicate.txt: 567 
 📄 case_education.txt: 877 
 📄 case_healthcare.txt: 547 
CPU times: user 859 µs, sys: 21 µs, total: 880 µs
Wall time: 5.96 ms


### 1: Fixed-size Chunking

In [23]:
%%time
def fixed_size_chunk(text, chunk_size=200, overlap=50):
 """"""
 chunks = []
 start = 0
 while start < len(text):
 end = start + chunk_size
 chunks.append(text[start:end])
 start = end - overlap # 
 return chunks

fixed_chunks = fixed_size_chunk(all_text, chunk_size=200, overlap=50)
print(f'📊 Fixed-size: {len(fixed_chunks)} chunks')
print(f'\n--- Chunk 1 ---')
print(fixed_chunks[0])
print(f'\n--- Chunk 2 ---')
print(fixed_chunks[1])

📊 Fixed-size: 18 chunks

--- Chunk 1 ---


: AI 

Smart Farming AI 
 IoT Sensor 

 Computer Vision 

--- Chunk 2 ---


 Computer Vision 
 Convolutional Neural Network (CNN) Train 
 50,000 8 

 NLP 
CPU times: user 205 µs, sys: 18 µs, total: 223 µs
Wall time: 231 µs


### 2: Recursive Character Chunking (LangChain)

 ** separator ** chunk → 

```python
separators = ['\n\n', '\n', '。', r'\. ', ' ', '']
# ① ② ③ ④ ⑤ ⑥
```

| | Separator | | |
|:-----:|-----------|----------|----------|
| ① | `\n\n` | **** () | |
| ② | `\n` | **** | |
| ③ | `。` | ** (/)** | CJK |
| ④ | `\. ` | ** + ** | "Hello. World" → |
| ⑤ | ` ` | **** | |
| ⑥ | `''` | **** () | |

**:**
```
 → \n\n 
 ↓ chunk 
 \n
 ↓ 
 
 ↓ ...
 ()
```

> 💡 ****: separator ` ` 
> `\n\n` `\n` 

In [24]:
%%time
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
 chunk_size=200,
 chunk_overlap=50,
 separators=['\n\n', '\n', '。', r'\. ', ' ', ''], # 
)

recursive_chunks = recursive_splitter.split_text(all_text)
print(f'📊 Recursive: {len(recursive_chunks)} chunks')
print(f'\n--- Chunk 1 ---')
print(recursive_chunks[0])
print(f'\n--- Chunk 2 ---')
print(recursive_chunks[1])

📊 Recursive: 17 chunks

--- Chunk 1 ---
: AI 

Smart Farming AI 
 IoT Sensor 

--- Chunk 2 ---
 Computer Vision 
 Convolutional Neural Network (CNN) Train 
 50,000 8 
CPU times: user 27.3 s, sys: 2.93 s, total: 30.2 s
Wall time: 55.6 s


### 3: Sentence-based Chunking

In [10]:
%%time
import re

def sentence_chunk(text, max_sentences=3):
 """/ ()"""
 # newline 
 sentences = re.split(r'\n+', text)
 sentences = [s.strip() for s in sentences if s.strip()]

 chunks = []
 for i in range(0, len(sentences), max_sentences):
 chunk = '\n'.join(sentences[i:i+max_sentences])
 if chunk:
 chunks.append(chunk)
 return chunks

sentence_chunks = sentence_chunk(all_text, max_sentences=3)
print(f'📊 Sentence-based: {len(sentence_chunks)} chunks')

# 3 chunks 
for idx in range(min(3, len(sentence_chunks))):
 print(f'\n--- Chunk {idx+1} ({len(sentence_chunks[idx])} ) ---')
 print(sentence_chunks[idx])

📊 Sentence-based: 18 chunks

--- Chunk 1 (150 ) ---
: AI 
Smart Farming AI 
 IoT Sensor 

--- Chunk 2 (166 ) ---
 Computer Vision 
 Convolutional Neural Network (CNN) Train 
 50,000 8 

--- Chunk 3 (143 ) ---
 NLP 
 Social Media

CPU times: user 761 µs, sys: 0 ns, total: 761 µs
Wall time: 769 µs


### 📊 Compare the Results


In [25]:
%%time
print('📊 Chunks:')
print(f' Fixed-size (200 chars): {len(fixed_chunks)} chunks')
print(f' Recursive (200 chars): {len(recursive_chunks)} chunks')
print(f' Sentence-based (3 sent): {len(sentence_chunks)} chunks')

print('\n📏 Chunk:')
for name, chunks in [('Fixed', fixed_chunks), ('Recursive', recursive_chunks), ('Sentence', sentence_chunks)]:
 sizes = [len(c) for c in chunks]
 print(f' {name:<12}: avg={np.mean(sizes):.0f}, min={min(sizes)}, max={max(sizes)} ')

📊 Chunks:
 Fixed-size (200 chars): 18 chunks
 Recursive (200 chars): 17 chunks
 Sentence-based (3 sent): 18 chunks

📏 Chunk:
 Fixed : avg=193, min=78, max=200 
 Recursive : avg=156, min=98, max=198 
 Sentence : avg=144, min=40, max=187 
CPU times: user 304 µs, sys: 3 µs, total: 307 µs
Wall time: 297 µs


### 💡 
- **Fixed-size** → 
- **Recursive** / → Fixed
- **Sentence-based** chunk 
- **overlap** 

> 🎯 : **Recursive** /

### 🧪 1.2

 `chunk_size` `overlap` ?

In [ ]:
%%time
# 🧪 : chunk_size overlap

# 💡 Hint:
# 1. chunk_size=100 vs 500 → chunks ?
# 2. overlap=0 vs 100 → ?

# for size in [100, 200, 500]:
# chunks = fixed_size_chunk(all_text, chunk_size=size, overlap=50)
# print(f'chunk_size={size}: {len(chunks)} chunks')

---
## 🤖 Section 1.3: Markdown Gemini

### Gemini Multimodal ?

**Multimodal** = + + PDF + video

```
 PDF → Gemini "" 
 ↓
 layout: header, , bullet, 
 ↓
 Markdown 
```

### vs 

| | | |
|---|------|--------|
| ✅ | layout | internet |
| ✅ | | API credits |
| ✅ | / | cloud |

> ⚠️ **** (, ) → Docling (Section 1.4)

In [12]:
%%time
# Gemini API
# 🔑 API Key : https://aistudio.google.com/apikey
#
# Colab Secrets:
# 1. 🔑 sidebar 
# 2. 'Add new secret'
# 3. : GOOGLE_API_KEY
# 4. API Key copy 
# 5. toggle 'Notebook access'

try:
 from google.colab import userdata
 GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets !')
except Exception:
 GOOGLE_API_KEY = input('🔑 Gemini API Key : ')

from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)

# API 
response = client.models.generate_content(
 model='gemini-2.5-flash',
 contents=' '
)
print(f'✅ Gemini API : {response.text.strip()}')

✅ API Key Colab Secrets !
✅ Gemini API : 
CPU times: user 7.83 s, sys: 446 ms, total: 8.28 s
Wall time: 28 s


### Sample PDF 

 PDF 

In [26]:
%%time
import fitz # PyMuPDF
import subprocess

# Thai font Colab
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-thai-tlwg'],
 capture_output=True)
print('✅ Thai font ')

# path font 
import glob
thai_fonts = glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf')
THAI_FONT = thai_fonts[0] if thai_fonts else None
print(f'📝 font: {THAI_FONT}')

def create_sample_pdf(filepath, text, title=''):
 """ PDF """
 doc = fitz.open()
 page = doc.new_page()

 # font 
 thai_font = fitz.Font(fontfile=THAI_FONT)

 # 
 tw = fitz.TextWriter(page.rect)
 y = 50
 tw.append((50, y), title, font=thai_font, fontsize=18)
 y += 40

 # 
 for line in text.split('\n'):
 if y > 750: # 
 tw.write_text(page)
 page = doc.new_page()
 tw = fitz.TextWriter(page.rect)
 y = 50
 tw.append((50, y), line, font=thai_font, fontsize=11)
 y += 18

 tw.write_text(page)
 doc.save(filepath)
 doc.close()

# PDF
with open('data/case_healthcare.txt', 'r', encoding='utf-8') as f:
 text = f.read()

create_sample_pdf('data/sample.pdf', text, ': AI ')
print('✅ sample.pdf (!)')

✅ Thai font 
📝 font: /usr/share/fonts/truetype/tlwg/Kinnari-BoldItalic.ttf
✅ sample.pdf (!)
CPU times: user 127 ms, sys: 17.1 ms, total: 144 ms
Wall time: 13.1 s


In [27]:
%%time
# Gemini PDF Markdown
try:
 # Gemini PDF Markdown
 import pathlib

 def pdf_to_markdown_gemini(pdf_path, client):
 """ Gemini PDF Markdown"""
 # PDF
 pdf_bytes = pathlib.Path(pdf_path).read_bytes()

 prompt = ''' PDF Markdown format
 :
 - (, , )
 - heading (#, ##, ###) 
 - Markdown table
 - bullet points
 - '''

 response = client.models.generate_content(
 model='gemini-2.5-flash',
 contents=[
 genai.types.Part.from_bytes(data=pdf_bytes, mime_type='application/pdf'),
 prompt
 ]
 )
 return response.text

 # 
 markdown_output = pdf_to_markdown_gemini('data/sample.pdf', client)
 print('📝 Markdown Gemini:')
 print('=' * 60)
 display(Markdown(markdown_output))
except Exception as e:
 print(f'❌ Error: {e}')
 print('💡 : 1) API Key 2) Internet 3) PDF ')


📝 Markdown Gemini:


# : AI 

## : AI 

 AI (Medical Imaging) X-ray Deep Learning AI 95% 92%

 NLP (EMR) 15 2 

: Transfer Learning Fine-tune 

CPU times: user 15.2 ms, sys: 2.01 ms, total: 17.2 ms
Wall time: 4.31 s


In [28]:
%%time
# 
with open('output/gemini_output.md', 'w', encoding='utf-8') as f:
 f.write(markdown_output)
print('✅ output/gemini_output.md')

✅ output/gemini_output.md
CPU times: user 1.14 ms, sys: 38 µs, total: 1.18 ms
Wall time: 1.27 ms


### 💡 
- Gemini ** layout** → header, , bullet 
- **Multimodal** → PDF "" 
- : internet + API

> ⚠️ (, ) — Docling (Section 1.4)

### 🧪 1.3

 upload PDF ( slides ) Gemini Markdown

In [ ]:
%%time
# 🧪 : PDF 
# TODO: Upload PDF pdf_to_markdown_gemini()



---
## 📄 Section 1.4: Markdown Docling

### Docling ?

**[Docling](https://github.com/DS4SD/docling)** open-source library IBM
 (PDF, DOCX, PPTX, HTML) Markdown JSON

| | Gemini | Docling |
|---|--------|--------|
| **** | Cloud API (LLM) | Local library |
| **** | quota | |
| **** | network | CPU/GPU |
| **** | | offline |
| **** | internet | |

In [29]:
%%time
from docling.document_converter import DocumentConverter

# PDF Docling
converter = DocumentConverter()
result = converter.convert('data/sample.pdf')

docling_markdown = result.document.export_to_markdown()

print('📝 Markdown Docling:')
print('=' * 60)
display(Markdown(docling_markdown))

[INFO] 2026-03-06 03:55:08,055 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:08,061 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:08,066 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.7.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:08,951 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-03-06 03:55:09,114 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,117 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-06 03:55:09,382 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-06 03:55:09,383 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-06 03:55:09,385 [RapidOCR] download_file.py:68: Initiating download: https://

📝 Markdown Docling:


## : AI 

: AI 

 AI (Medical Imaging) X-ray Deep Learning AI 95% 92%

 NLP (EMR) 15 2 

: Transfer Learning Fine-tune 

CPU times: user 13 s, sys: 6.69 s, total: 19.7 s
Wall time: 35.3 s


In [30]:
%%time
# 
with open('output/docling_output.md', 'w', encoding='utf-8') as f:
 f.write(docling_markdown)

print('📊 Gemini vs Docling:')
print(f' Gemini output: {len(markdown_output)} ')
print(f' Docling output: {len(docling_markdown)} ')
print('\n💡 Tip: ')
print(' - (, ) → Gemini')
print(' - , offline → Docling')

📊 Gemini vs Docling:
 Gemini output: 581 
 Docling output: 579 

💡 Tip: 
 - (, ) → Gemini
 - , offline → Docling
CPU times: user 1.29 ms, sys: 0 ns, total: 1.29 ms
Wall time: 989 µs


### 📊 Gemini vs Docling

| | Gemini API | Docling (IBM) |
|-------|-----------|---------------|
| **** | ⚡ ( API) | 🐢 (run local) |
| ** Markdown** | ⭐⭐⭐ | ⭐⭐ |
| **** | ✅ | ⚠️ |
| **** | 💰 ( quota) | 🆓 |
| ** Internet** | ✅ | ❌ |
| **/** | ✅ | ✅ |
| **** | ⚠️ cloud | ✅ |
| **** | Prototype, | Production, |

### 🧪 1.4

 PDF Gemini Docling 

In [ ]:
%%time
# 🧪 : Gemini vs Docling PDF 
# TODO: 



---
## 🔢 Section 1.5: Embedding ()

### Embedding ?

**Embedding = (vector)** 

```
"AI " → [0.23, -0.11, 0.45, ...] (1024 )
"" → [0.21, -0.13, 0.42, ...] ← ! ()
"" → [0.89, 0.56, -0.32, ...] ← ()
```

### multilingual?

| Model | | Vector Size | |
|-------|:-------:|:-----------:|:--------:|
| `all-MiniLM-L6-v2` | ❌ | 384 | ⚡ |
| `multilingual-e5-large` | ✅ | 1024 | 🐢 |
| `paraphrase-multilingual` | ✅ | 768 | ⚡ |

> 💡 **multilingual-e5-large** 
> ( fine-tune model )

In [31]:
%%time
# Embedding Model ( HuggingFace)
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

try:
 embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
 test = embed_model.encode('')
 print(f'✅ Embedding Model !')
 print(f' Model: intfloat/multilingual-e5-large')
 print(f' Vector size: {len(test)}')
except Exception as e:
 print(f'❌ Model : {e}')
 print('💡 : 1) Internet 2) Disk space ( ~2GB)')

✅ Embedding Model !
 Model: intfloat/multilingual-e5-large
 Vector size: 1024
CPU times: user 12.5 s, sys: 16.2 s, total: 28.7 s
Wall time: 38.4 s


In [33]:
%%time
# embedding 
thai_sentences = [
 'query: ',
 'passage: AI ',
 'passage: Machine Learning AI ',
 'passage: ',
 'passage: Vector Database ',
]

# embeddings
embeddings = embed_model.encode(thai_sentences)

print(f'📊 embedding !')
print(f' : {len(embeddings)}')
print(f' vector: {embeddings.shape[1]} ')
print(f' vector (5 ): {embeddings[0][:5]}')

📊 embedding !
 : 5
 vector: 1024 
 vector (5 ): [ 0.02122011 -0.01509202 0.01514856 -0.05054271 0.05155794]
CPU times: user 2.29 s, sys: 36.2 ms, total: 2.32 s
Wall time: 1.52 s


### 📐 Vectors

 Vector **** vector 
 3 :

---

**1️⃣ Cosine Similarity** — "" 2 vectors
```
 A · B (a₁×b₁ + a₂×b₂ + ...)
Cosine(A, B) = ─────────── = ─────────────────────────────────────────
 |A| × |B| A × B

: -1 () ← 0 () → 1 () ✅
```

**2️⃣ Dot Product** — 
```
Dot(A, B) = a₁×b₁ + a₂×b₂ + a₃×b₃ + ...

: -∞ ← 0 → +∞ ( = )
```

**3️⃣ L2 Distance (Euclidean)** — "" 2 
```
L2(A, B) = √( (a₁-b₁)² + (a₂-b₂)² + (a₃-b₃)² + ... )

: 0 () → +∞ () ⚠️ = 
```

---

### 🤔 ?

**1️⃣ Cosine Similarity — RAG / NLP**

 "" "" vector

```
: 
 A: AI 10 → vector ()
 B: AI 1 → vector ()

 Cosine = 0.95 ✅ → "" 
 Dot Product = ❌ → vector 
```
👉 ****: , RAG retrieval, semantic search

---

**2️⃣ Dot Product — vector normalize **

 normalize vector = 1 → Dot Product = Cosine ( )

```
: recommendation
 User vector (normalize): [0.5, 0.7, 0.1] ( = 1)
 Product A vector (normalize): [0.4, 0.8, 0.2] ( = 1)

 Dot = 0.5×0.4 + 0.7×0.8 + 0.1×0.2 = 0.78 ← !
```
👉 ****: , vector normalize , production 

---

**3️⃣ L2 Distance — "" **

 

```
: (Image Clustering)
 A: [0.2, 0.8, 0.3]
 B: [0.3, 0.7, 0.4] → L2 = 0.17 ( ✅)
 : [0.9, 0.1, 0.8] → L2 = 1.12 ( ❌)
```
👉 ****: image similarity, clustering, anomaly detection

---

> 💡 ****: RAG → **Cosine Similarity** !

In [ ]:
%%time
# ─── Thai font matplotlib ───
import subprocess, glob
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Thai font package ( Colab)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-thai-tlwg'], capture_output=True)

# register font matplotlib
for font_path in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
 fm.fontManager.addfont(font_path)

plt.rcParams['font.family'] = 'Garuda'
print('✅ Thai font (Garuda) ')

In [34]:
%%time
# === 3 — Heatmap ===
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

labels = ['Q: AI ', 'P: AI CS', 'P: ML AI', 'P: ', 'P: VectorDB']
short_labels = ['Q: AI?', 'P: AI-CS', 'P: ML-AI', 'P: ', 'P: VecDB']

# 3 
cos_sim = cosine_similarity(embeddings)
dot_prod = np.dot(embeddings, embeddings.T)
l2_dist = euclidean_distances(embeddings)

# --- Heatmap Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

matrices = [
 (cos_sim, '1️⃣ Cosine Similarity\n( = )', 'YlOrRd', '.3f'),
 (dot_prod, '2️⃣ Dot Product\n( = )', 'YlOrRd', '.1f'),
 (l2_dist, '3️⃣ L2 Distance\n( = )', 'YlOrRd_r', '.3f'),
]

for ax, (matrix, title, cmap, fmt) in zip(axes, matrices):
 im = ax.imshow(matrix, cmap=cmap, aspect='auto')
 ax.set_xticks(range(len(short_labels)))
 ax.set_yticks(range(len(short_labels)))
 ax.set_xticklabels(short_labels, fontsize=10, rotation=45, ha='right')
 ax.set_yticklabels(short_labels, fontsize=10)
 ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

 # 
 for i in range(len(labels)):
 for j in range(len(labels)):
 val = matrix[i][j]
 color = 'white' if val > (matrix.max() + matrix.min()) / 2 else 'black'
 ax.text(j, i, f'{val:{fmt}}', ha='center', va='center', fontsize=9, color=color)

 plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('📐 3 Vector', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n💡 :')
print(' - Cosine: AI similarity (~0.9), (~0.7)')
print(' - L2: AI , ')
print(' - Dot Product Cosine vector normalize ')

1️⃣ Cosine Similarity ( = , max=1.0)
 Q: AI P: AI CSP: ML AI P: P: VectorDB
 Q: AI 1.000 0.850 0.818 0.722 0.754 
 P: AI CS 0.850 1.000 0.917 ⭐ 0.796 0.827 
P: ML AI 0.818 0.917 ⭐ 1.000 0.806 0.846 
 P: 0.722 0.796 0.806 1.000 0.818 
 P: VectorDB 0.754 0.827 0.846 0.818 1.000 


2️⃣ Dot Product ( = , max)
 Q: AI P: AI CSP: ML AI P: P: VectorDB
 Q: AI 1.0 0.8 0.8 0.7 0.8
 P: AI CS 0.8 1.0 0.9 0.8 0.8
P: ML AI 0.8 0.9 1.0 0.8 0.8
 P: 0.7 0.8 0.8 1.0 0.8
 P: VectorDB 0.8 0.8 0.8 0.8 1.0


3️⃣ L2 Distance ( = , min=0.0)
 Q: AI P: AI CSP: ML AI P: P: VectorDB
 Q: AI 0.000 0.548 ⭐ 0.603 0.746 0.702 
 P: AI CS 0.548 ⭐ 0.000 0.408 ⭐ 0.638 0.588 ⭐
P: ML AI 0.603 0.408 ⭐ 0.000 0.623 0.554 ⭐
 P: 0.746 0.638 0.623 0.000 0.603 
 P: VectorDB 0.702 0.588 ⭐ 0.554 ⭐ 0.603 0.000 

💡 :
 - Cosine: AI similarity (~0.9), (~0.7)
 - L2: AI , 
 - Dot Product Cosine vector normalize 
CPU times: user 3.82 ms, sys: 1.94 ms, total: 5.76 ms
Wall time: 16.3 ms


### 🎨 Visualization: = 

 1024 → 2 PCA plot vector 

In [ ]:
%%time
from sklearn.decomposition import PCA

# 1024 → 2 PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

# Plot
plt.figure(figsize=(10, 7))

labels = ['Q: AI ', 'P: AI CS', 'P: ML AI', 'P: ', 'P: VectorDB']
colors = ['red', 'blue', 'blue', 'gray', 'green']

for j, (x, y) in enumerate(coords):
 plt.scatter(x, y, c=colors[j], s=200, zorder=5)
 plt.annotate(labels[j], (x, y), textcoords='offset points',
 xytext=(10, 10), fontsize=11,
 bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# query passages
for j in range(1, len(coords)):
 plt.plot([coords[0][0], coords[j][0]], [coords[0][1], coords[j][1]],
 'k--', alpha=0.3)

plt.title('📐 Embedding Space (PCA 2D) — = ', fontsize=14)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('💡 : AI () ')
print(' () ')

### 💡 
- **** → cosine score ** 1.0**
- **** → cosine score ** 0**
- PCA plot: **** → semantic search
- `multilingual-e5-large` train 100 

> 🎯 **Embedding RAG** — embedding 

### 🧪 1.5

 5 similarity 

In [ ]:
%%time
# 🧪 : embedding 
# TODO: 



---
## 🔀 Section 1.6: Hybrid Embedding (Dense + Sparse)

### Dense Embedding ?

**Dense Embedding** vector **** ( 0)
 Neural Network "" 

```
"" → [0.12, -0.34, 0.56, 0.78, 0.23, -0.11, ...] (1024 , 0)
```

✅ ****: "" — "" "" 
❌ ****: keyword "SKU-12345"

---

### Sparse Embedding ?

**Sparse Embedding** vector ** 0**
 BM25 TF-IDF

```
 vocabulary = [, , , , , , ...] (50,000 )

"" → [0.8, 0, 0.5, 0, 0.3, 0, 0, 0, 0, ...] (50,000 , 0)
 
```

✅ ****: keyword , , 
❌ ****: synonym — "" "" 

---

### 🔀 Hybrid Search — ?

| | Dense | Sparse | Hybrid |
|-----------|-------|--------|--------|
| "AI " → "" | ✅ | ❌ | ✅ |
| "SKU-12345" | ❌ | ✅ | ✅ |
| "" | ✅ | ✅ | ✅✅ |

** Hybrid:**
```
Hybrid Score = α × Dense Score + (1-α) × Sparse Score

α = 1.0 → Dense 
α = 0.5 → ()
α = 0.0 → Sparse 
```

### 🇹🇭 Tokenizer ?

 **** :

```
English: "I love machine learning"
 → split → ["I", "love", "machine", "learning"] ✅ !

Thai: ""
 → split → [""] ❌ !
 → pythainlp → ["", "", "", "", "", ""] ✅
```

**BM25 token () ** → !

 `pythainlp.word_tokenize()` dictionary + ML 

In [36]:
%%time
from rank_bm25 import BM25Okapi
import pythainlp

# 
documents = [
 ' ',
 'Machine Learning AI ',
 'NLP ',
 'Vector Database ',
 'RAG LLM ',
 'Qdrant open-source vector database',
]

# PyThaiNLP
tokenized_docs = [pythainlp.word_tokenize(doc) for doc in documents]
print('📝 :')
for i, (doc, tokens) in enumerate(zip(documents[:2], tokenized_docs[:2])):
 print(f' [{i}] {doc}')
 print(f' → {tokens}')

# BM25 index (Sparse)
bm25 = BM25Okapi(tokenized_docs)
print(f'\n✅ BM25 index ({len(documents)} documents)')

📝 :
 [0] 
 → ['', ' ', '', '', '', '', '', '']
 [1] Machine Learning AI 
 → ['Machine', ' ', 'Learning', ' ', '', '', '', ' ', 'AI', ' ', '', '', '', '']

✅ BM25 index (6 documents)
CPU times: user 2.82 s, sys: 235 ms, total: 3.05 s
Wall time: 3.15 s


In [37]:
%%time
# Hybrid Search
def hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5, top_k=3):
 """ Hybrid: Dense + Sparse"""
 # Dense search (Semantic)
 q_emb = embed_model.encode([f'query: {query}'])
 doc_embs = embed_model.encode([f'passage: {d}' for d in documents])
 dense_scores = cosine_similarity(q_emb, doc_embs)[0]

 # Sparse search (BM25)
 q_tokens = pythainlp.word_tokenize(query)
 sparse_scores = bm25.get_scores(q_tokens)

 # Normalize scores to [0, 1]
 if max(dense_scores) > 0:
 dense_norm = dense_scores / max(dense_scores)
 else:
 dense_norm = dense_scores

 if max(sparse_scores) > 0:
 sparse_norm = sparse_scores / max(sparse_scores)
 else:
 sparse_norm = sparse_scores

 # Hybrid score
 hybrid_scores = alpha * dense_norm + (1 - alpha) * sparse_norm

 # 
 ranked = sorted(enumerate(zip(documents, dense_norm, sparse_norm, hybrid_scores)),
 key=lambda x: x[1][3], reverse=True)
 return ranked[:top_k]

# 
query = 'AI '
results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=0.5)

print(f'🔍 Query: "{query}"')
print(f'{"":>4}{"Document":<50} {"Dense":>8} {"Sparse":>8} {"Hybrid":>8}')
print('=' * 82)
for idx, (doc, dense, sparse, hybrid) in results:
 print(f' [{idx}] {doc:<48} {dense:>8.3f} {sparse:>8.3f} {hybrid:>8.3f}')

🔍 Query: "AI "
 Document Dense Sparse Hybrid
 [1] Machine Learning AI 1.000 1.000 1.000
 [5] Qdrant open-source vector database 0.919 0.138 0.528
 [3] Vector Database 0.922 0.110 0.516
CPU times: user 2.79 s, sys: 23.6 ms, total: 2.81 s
Wall time: 2.49 s


In [38]:
%%time
# alpha 
print('📊 alpha (Dense weight) :')
print('=' * 70)
query = 'vector database RAG'
print(f'Query: "{query}"\n')

for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
 results = hybrid_search(query, documents, embed_model, bm25, tokenized_docs, alpha=alpha, top_k=2)
 top_doc = results[0][1][0]
 top_score = results[0][1][3]
 label = 'Sparse only' if alpha == 0 else 'Dense only' if alpha == 1 else f'Hybrid'
 print(f' α={alpha:.1f} ({label:<12}): [{results[0][0]}] {top_doc[:45]}... ({top_score:.3f})')

📊 alpha (Dense weight) :
Query: "vector database RAG"

 α=0.0 (Sparse only ): [5] Qdrant open-source vector database... (1.000)
 α=0.3 (Hybrid ): [5] Qdrant open-source vector database... (0.994)
 α=0.5 (Hybrid ): [5] Qdrant open-source vector database... (0.991)
 α=0.7 (Hybrid ): [5] Qdrant open-source vector database... (0.987)
 α=1.0 (Dense only ): [3] Vector Database ... (1.000)
CPU times: user 13.8 s, sys: 228 ms, total: 14 s
Wall time: 14.4 s


### 💡 
- **Dense** (embedding) **** → "" = ""
- **Sparse** (BM25) **** → "Python" 
- **alpha=0.7** (Dense 70% + Sparse 30%) 
- tokenizer (PyThaiNLP) 

> 🎯 **Hybrid = + ** → 

### 🧪 1.6

 query alpha 

In [ ]:
%%time
# 🧪 : Hybrid search query 

# 💡 Hint:
# 1. alpha : 0.0 (Sparse only), 0.5 (50/50), 1.0 (Dense only)
# 2. query keyword vs 

# hybrid_search('Deep Learning', all_chunks, embed_model, bm25,
# tokenized_docs, alpha=0.5, top_k=3)

---
## 🗄️ Section 1.7: Qdrant VectorDB

### Vector Database ?

**Vector Database** = vector 

```
 → Embedding → Vector [0.12, -0.45, 0.78, ...] → VectorDB
 ↕
Query → Embedding → Vector → vector (ANN)
```

### Qdrant?

| VectorDB | | Cloud | In-Memory | Open Source |
|----------|------|:-----:|:---------:|:-----------:|
| **Qdrant** | Rust | ✅ | ✅ | ✅ |
| ChromaDB | Python | ❌ | ✅ | ✅ |
| Pinecone | — | ✅ | ❌ | ❌ |
| Weaviate | Go | ✅ | ✅ | ✅ |

> 💡 **Qdrant** (Rust), in-memory ( Colab), filter payload 

In [44]:
%%time
from qdrant_client import QdrantClient, models

try:
 qdrant = QdrantClient(':memory:')
 print('✅ Qdrant ! (in-memory mode)')
 print('💡 in-memory = RAM Colab ')
except Exception as e:
 print(f'❌ Qdrant: {e}')

✅ Qdrant ! (in-memory mode)
💡 in-memory = RAM Colab 
CPU times: user 10.1 ms, sys: 0 ns, total: 10.1 ms
Wall time: 12.1 ms


In [47]:
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

In [49]:
%%time
COLLECTION_NAME = 'thai_documents'
VECTOR_SIZE = 1024

# Collection 
if qdrant.collection_exists(collection_name=COLLECTION_NAME):
 qdrant.delete_collection(collection_name=COLLECTION_NAME)
 print(f'🗑️ Collection "{COLLECTION_NAME}" ')

# Collection
qdrant.create_collection(
 collection_name=COLLECTION_NAME,
 vectors_config=models.VectorParams(
 size=VECTOR_SIZE,
 distance=models.Distance.COSINE, # Cosine similarity
 ),
)

# 
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Collection "{COLLECTION_NAME}" !')
print(f' Status: {info.status}')
print(f' Vector size: {info.config.params.vectors.size}')
print(f' Distance: {info.config.params.vectors.distance}')
print(f' Points: {info.points_count}')

🗑️ Collection "thai_documents" 
✅ Collection "thai_documents" !
 Status: green
 Vector size: 1024
 Distance: Cosine
 Points: 0
CPU times: user 999 µs, sys: 0 ns, total: 999 µs
Wall time: 1.01 ms


### 💡 
- `:memory:` = → Colab 
- `Cosine` = "" vector → NLP
- Production → Qdrant Cloud Docker + persistent storage

> 🎯 VectorDB ** vectors milliseconds** ANN algorithm

### 🧪 1.7

 collection `my_collection` distance metric `EUCLID`

In [ ]:
# 🧪 : collection 
# TODO: 



---
## 📥 Section 1.8: Index (Upsert )

### Indexing ?

**Indexing** = chunk + embedding + metadata VectorDB

```
 chunk:
 1. Embed: chunk → vector
 2. Point: id + vector + payload (text, source)
 3. Upsert: Qdrant ( id update)
```

| | |
|--------|----------|
| **Point** | Qdrant (id + vector + payload) |
| **Payload** | metadata vector (text, source, date, ...) |
| **Upsert** | Insert / Update |

In [50]:
%%time
# indexing
# → chunk recursive splitter

all_chunks = []
for fname in sorted(os.listdir('data')):
 if not fname.endswith('.txt'):
 continue
 with open(f'data/{fname}', 'r', encoding='utf-8') as f:
 text = f.read()

 chunks = recursive_splitter.split_text(text)
 for i, chunk in enumerate(chunks):
 all_chunks.append({
 'text': chunk,
 'source': fname,
 'chunk_index': i,
 })

print(f'📊 :')
print(f' chunks : {len(all_chunks)}')
for fname in sorted(os.listdir('data')):
 if fname.endswith('.txt'):
 count = sum(1 for c in all_chunks if c['source'] == fname)
 print(f' {fname}: {count} chunks')

📊 :
 chunks : 18
 case_agriculture.txt: 4 chunks
 case_banking_duplicate.txt: 4 chunks
 case_education.txt: 6 chunks
 case_healthcare.txt: 4 chunks
CPU times: user 0 ns, sys: 1.51 ms, total: 1.51 ms
Wall time: 1.39 ms


In [42]:
%%time
# embeddings chunk
print('⏳ embeddings...')
chunk_texts = [f"passage: {c['text']}" for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, show_progress_bar=True)
print(f'✅ embeddings ! ({len(chunk_embeddings)} vectors × {chunk_embeddings.shape[1]} )')

⏳ embeddings...


Batches: 0%| | 0/1 [00:00<?, ?it/s]

✅ embeddings ! (18 vectors × 1024 )
CPU times: user 18.7 s, sys: 266 ms, total: 18.9 s
Wall time: 11.5 s


In [51]:
%%time
# Upsert Qdrant
points = []
for i, (chunk, embedding) in enumerate(zip(all_chunks, chunk_embeddings)):
 points.append(
 models.PointStruct(
 id=i,
 vector=embedding.tolist(),
 payload={
 'text': chunk['text'],
 'source': chunk['source'],
 'chunk_index': chunk['chunk_index'],
 }
 )
 )

# Upsert 
qdrant.upsert(
 collection_name=COLLECTION_NAME,
 points=points,
)

# 
info = qdrant.get_collection(COLLECTION_NAME)
print(f'✅ Upsert ! points collection: {info.points_count}')

✅ Upsert ! points collection: 18
CPU times: user 19.8 ms, sys: 814 µs, total: 20.6 ms
Wall time: 21.1 ms


### 💡 
- **payload** = metadata vector ( text, source)
- filter `source='healthcare'`
- **upsert** = insert + update → id update 

> 🎯 payload → filter 

### 🧪 1.8

 collection ( )

In [ ]:
# 🧪 : 
# TODO: 



---
## 🔎 Section 1.9: Retrieve Data from Qdrant

### What is Retrieval?

**Retrieval** = finding chunks relevant to a query from the VectorDB.

```
Query: "How does AI help doctors?"
 ↓ embed
Vector: [0.23, -0.11, ...]
 ↓ search in Qdrant (Cosine Similarity)
Top-3 Results:
 1. [score=0.92] "Siriraj Hospital uses AI to analyze X-ray images..."
 2. [score=0.85] "NLP analyzes electronic medical records..."
 3. [score=0.71] "Deep Learning is used to detect cancer..."
```

### Tunable settings

| Parameter | Function | Recommended value |
|-----------|--------|----------|
| `top_k` | Number of chunks to retrieve | 3-5 (balance precision/coverage) |
| `score_threshold` | Cut off low-score results | 0.5+ |
| `filter` | Filter by metadata | source, date, category |



### Dense Search (Semantic Search)

### 🎯 top_k ?

**`top_k`** score 

```
: 100 chunks database

top_k=3 → 3 chunks ✅ , 
top_k=5 → 5 chunks 
top_k=10 → 10 chunks ⚠️ noise 
```

** top_k :**

| top_k | | | |
|-------|---------|-------|--------|
| 1-3 | | , | |
| 3-5 | () | | — |
| 5-10 | / | | , context |

> 💡 ****: top_k=3~5 RAG
> LLM 

In [52]:
%%time
def search_qdrant(query, top_k=3):
 """ Qdrant Dense embedding"""
 # query embedding
 q_embedding = embed_model.encode([f'query: {query}'])[0]

 # 
 results = qdrant.query_points(
 collection_name=COLLECTION_NAME,
 query=q_embedding.tolist(),
 limit=top_k,
 with_payload=True,
 )
 return results

# 
query = ''
results = search_qdrant(query)

print(f'🔍 Query: "{query}"')
print(f'📊 Top-3:')
print('=' * 70)
for i, point in enumerate(results.points):
 print(f'\n🏆 {i+1} (Score: {point.score:.4f})')
 print(f' : {point.payload["source"]} (chunk {point.payload["chunk_index"]})')
 print(f' : {point.payload["text"][:150]}...')

🔍 Query: ""
📊 Top-3:

🏆 1 (Score: 0.7913)
 : case_agriculture.txt (chunk 0)
 : : AI 

Smart Farming AI 
 IoT Sensor ...

🏆 2 (Score: 0.7865)
 : case_healthcare.txt (chunk 0)
 : : AI ...

🏆 3 (Score: 0.7815)
 : case_healthcare.txt (chunk 1)
 : AI (Medical Imaging)
 X-ray Deep Learning
...
CPU times: user 543 ms, sys: 14.6 ms, total: 558 ms
Wall time: 369 ms


### Try Searching with Other Questions


In [53]:
%%time
# 
test_queries = [
 'Vector Database ',
 'RAG ',
 'Deep Learning Machine Learning ',
]

for query in test_queries:
 results = search_qdrant(query, top_k=2)
 print(f'\n🔍 Query: "{query}"')
 for i, point in enumerate(results.points):
 print(f' [{i+1}] (Score: {point.score:.4f}) {point.payload["text"][:80]}...')
 print('-' * 70)


🔍 Query: "Vector Database "
 [1] (Score: 0.8410) : Vector Database Embedding
Hybrid Search Dense + ...
 [2] (Score: 0.8151) 4. : Upsert Vector Qdrant Vector DB 
5. : Hybrid S...
----------------------------------------------------------------------

🔍 Query: "RAG "
 [1] (Score: 0.8712) RAG 
 ...
 [2] (Score: 0.8572) RAG -
 ...
----------------------------------------------------------------------

🔍 Query: "Deep Learning Machine Learning "
 [1] (Score: 0.8206) : Vector Database Embedding
Hybrid Search Dense + ...
 [2] (Score: 0.8100) 4. : Upsert Vector Qdrant Vector DB 
5. : Hybrid S...
----------------------------------------------------------------------
CPU times: user 1.57 s, sys: 33.1 ms, total: 1.6 s
Wall time: 1.03 s


### Filter Metadata

In [54]:
%%time
# 
query = ''
q_embedding = embed_model.encode([f'query: {query}'])[0]

results_filtered = qdrant.query_points(
 collection_name=COLLECTION_NAME,
 query=q_embedding.tolist(),
 query_filter=models.Filter(
 must=[
 models.FieldCondition(
 key='source',
 match=models.MatchValue(value='case_healthcare.txt'),
 )
 ]
 ),
 limit=3,
 with_payload=True,
)

print(f'🔍 Query: "{query}" (filter: source=case_healthcare.txt)')
for i, point in enumerate(results_filtered.points):
 print(f' [{i+1}] ({point.score:.4f}) [{point.payload["source"]}] {point.payload["text"][:80]}...')

🔍 Query: "" (filter: source=case_healthcare.txt)
 [1] (0.7978) [case_healthcare.txt] : AI ...
 [2] (0.7856) [case_healthcare.txt] : 
 Transfer Learning...
 [3] (0.7817) [case_healthcare.txt] AI (Medical Imaging)...
CPU times: user 486 ms, sys: 10.2 ms, total: 496 ms
Wall time: 312 ms


### 💡 
- score **1.0** = , **0** = 
- query ** text** → "AI " "" 
- `top_k` (3-5) → | (10-20) → noise
- **Filter** → + 

> 🎯 **Retrieval RAG** — = 

### 🧪 1.9

1. 
2. filter 
3. `top_k` 

In [ ]:
%%time
# 🧪 : Qdrant

# 💡 Hint:
# 1. query vs 
# 2. top_k → ?
# 3. filter source 

# search_qdrant('', top_k=5)
# search_qdrant('AI', top_k=3) # ?

---
## 🚀 Bonus: End-to-End RAG Demo

 retrieved chunks Gemini !

```
 → Retrieval (Qdrant) → Top Chunks → LLM (Gemini) → 
```

> 💡 **Day 2** Agent !

In [56]:
%%time
def rag_answer(question, top_k=3):
 """ RAG : + """
 # 1. Embed query
 query_vector = embed_model.encode(f'query: {question}').tolist()

 # 2. Retrieve from Qdrant
 results = qdrant.query_points(
 collection_name='thai_documents',
 query=query_vector,
 limit=top_k
 ).points

 # 3. context chunks 
 context_parts = []
 print(f'🔍 : {question}')
 print(f'📚 Retrieved {len(results)} chunks:')
 for idx, r in enumerate(results):
 text = r.payload['text']
 context_parts.append(text)
 print(f' [{idx+1}] score={r.score:.4f} | {text[:60]}...')

 context = '\n\n'.join(context_parts)

 # 4. Gemini 
 prompt = f''' 
 ""

:
{context}

: {question}
:'''

 response = client.models.generate_content(
 model='gemini-2.5-flash',
 contents=prompt
 )

 print(f'\n🤖 RAG:')
 print(f' {response.text.strip()}')
 print()
 return response.text.strip()

# 
questions = [
 'AI ',
 'RAG ',
 ' AI ',
 ' AI '
]

for q in questions:
 print('=' * 60)
 rag_answer(q)
 print()

🔍 : AI 
📚 Retrieved 3 chunks:
 [1] score=0.8733 | AI ...
 [2] score=0.8680 | : AI ...
 [3] score=0.8512 | : AI 

...

🤖 RAG:
 AI X-ray 


🔍 : RAG 
📚 Retrieved 3 chunks:
 [1] score=0.8503 | : AI 

...
 [2] score=0.8480 | RAG -
...
 [3] score=0.8456 | RAG 
...

🤖 RAG:
 RAG ( Chatbot KBTG) FAQ LLM 


🔍 : AI 
📚 Retrieved 3 chunks:
 [1] score=0.8596 | :
1. : PDF ...
 [2] score=0.8370 | : AI 

...
 [3] score=0.8259 | 4. : Upsert Vector Qdrant Vector DB 
5....

🤖 RAG:
 


🔍 : AI 
📚 Retrieved 3 chunks:
 [1] score=0.8675 | : AI 

...
 [2] score=0.8349 | 
...
 [3] score=0.8272 | AI ...

🤖 RAG:
 AI Intelligent Tutoring System LLM 


CPU times: user 1.99 s, sys: 48.3 ms, total: 2.04 s
Wall time: 18.4 s


---
## 🎯 : 

### Pipeline Data Engineering RAG

| Step | Section | Task |
|:----:|---------|------|
| 📄 | Input | Raw Documents (PDF, TXT, DOCX) |
| 🔍 | 1.1 | Duplicate (MD5) → |
| 📝 | 1.3-1.4 | Markdown (Gemini / Docling) |
| ✂️ | 1.2 | Chunking (Fixed / Recursive / Sentence) |
| 🔢 | 1.5 | Dense Embedding (multilingual-e5-large) |
| 🔀 | 1.6 | Hybrid Embedding (Dense + BM25) |
| 🗄️ | 1.7 | Qdrant Collection |
| 📥 | 1.8 | Index (Upsert vectors + metadata) |
| 🔎 | 1.9 | Retrieve (Dense / Filtered search) |


### 🔑 Key Takeaways

1. **** — "Garbage in, garbage out"
2. **Chunking strategy** RAG
3. **Hybrid search** (Dense + Sparse) 
4. **Metadata** filter 

### 📅 : Day 2 — Building Agents: Finding the Needle in the Haystack

 **AI Agents** RAG pipeline 
!